# Customer Behavior & Retention Analysis

## Business problem
Leadership wants to know who the best customers are, why products are returned, and where to spend marketing money.

# 1. Setup & Load Data

In [1]:
# import libraries
import pandas as pd
import numpy as np


# import data
customers_raw = pd.read_csv('customers_raw.csv')
orders_raw = pd.read_csv('orders_raw.csv')


In [2]:
# create copy of the original dataset
customers = customers_raw.copy()
orders = orders_raw.copy()

# 2. Cleaning - Customers table


## 2.1 Audit (find ALL problems first, fix nothing yet)

In [3]:
customers.head()

,customer_id,gender,age,city,signup_date,acquisition_channel,is_prime_member,state
0,C1000,male,55.0,Delhi,2023-03-31,Email,No,Delhi
1,C1001,Female,22.0,Delhi,2025-02-12,Google Ads,No,Delhi
2,C1002,Female,33.0,Indore,2024-10-16,Organic Search,No,Madhya Pradesh
3,C1003,Female,20.0,Pune,2024-03-14,Google Ads,No,Maharashtra
4,C1004,Female,24.0,Bengaluru,2024-03-09,Instagram Ads,No,Karnataka


In [4]:
customers.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1200 entries, 0 to 1199
Data columns (total 8 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   customer_id          1200 non-null   object 
 1   gender               1200 non-null   object 
 2   age                  1130 non-null   float64
 3   city                 1143 non-null   object 
 4   signup_date          1200 non-null   object 
 5   acquisition_channel  1200 non-null   object 
 6   is_prime_member      1200 non-null   object 
 7   state                1200 non-null   object 
dtypes: float64(1), object(7)
memory usage: 75.1+ KB


In [5]:
customers.describe(include = "all")

,customer_id,gender,age,city,signup_date,acquisition_channel,is_prime_member,state
count,1200,1200,1130.000000,1143,1200,1200,1200,1200
unique,1200,11,NaN,15,700,6,2,10
top,C2183,Male,NaN,Mumbai,2024-03-09,Organic Search,No,Maharashtra
freq,1,524,NaN,153,6,321,818,303
mean,NaN,NaN,36.545133,NaN,NaN,NaN,NaN,NaN
std,NaN,NaN,64.853150,NaN,NaN,NaN,NaN,NaN
min,NaN,NaN,18.000000,NaN,NaN,NaN,NaN,NaN
25%,NaN,NaN,25.000000,NaN,NaN,NaN,NaN,NaN
50%,NaN,NaN,32.000000,NaN,NaN,NaN,NaN,NaN
75%,NaN,NaN,39.000000,NaN,NaN,NaN,NaN,NaN


In [6]:
customers.duplicated().sum()

np.int64(0)

In [7]:
for col in ['gender', 'city', 'acquisition_channel', 'is_prime_member']:
    print(customers[col].value_counts(dropna=False), '\n')

gender
Male      524
Female    506
male       43
female     43
Other      42
F          18
M          18
other       2
O           2
f           1
m           1
Name: count, dtype: int64 

city
Mumbai       153
Bengaluru    149
Delhi        145
Chennai       99
Hyderabad     95
Pune          82
Kolkata       78
Jaipur        58
NaN           57
Ahmedabad     51
Bangalore     50
Lucknow       46
Indore        42
Nagpur        40
Bhopal        35
Ujjain        20
Name: count, dtype: int64 

acquisition_channel
Organic Search    321
Instagram Ads     282
Google Ads        241
Referral          132
Email             124
YouTube           100
Name: count, dtype: int64 

is_prime_member
No     818
Yes    382
Name: count, dtype: int64 



In [8]:
customers['signup_date'] = pd.to_datetime(customers['signup_date'])

In [9]:
customers['signup_date'].dtype

dtype('<M8[ns]')

In [10]:
customers['signup_date'].describe()

,signup_date
count,1200
mean,2024-05-18 17:40:48
min,2023-01-05 00:00:00
25%,2023-09-11 18:00:00
50%,2024-05-20 12:00:00
75%,2025-01-26 06:00:00
max,2025-09-26 00:00:00


In [11]:
customers.columns.to_list()

['customer_id',
 'gender',
 'age',
 'city',
 'signup_date',
 'acquisition_channel',
 'is_prime_member',
 'state']

## 2.2 Issue register - Customers table


| # | Column | Problem found | Planned fix |
|---|---|---|---|
| 1 | Customers | 57 Null city | Fill with 'Unknown' |
| 2 | Customers | Age Has Outliers | Remove Outlier |
| 3 | Customers | 70 Null Age | Fill with Median & Convert to int|
| 4 | Customers | Gender has 11 unique Value | Convert standared "Male", "Female", "Others" |
| 5 | Customers | bangalore wrong spell | Convert to Bengaluru |

## 2.3 Fixes - Customers table



**Issue 1** :
57 Null City -> Fill With 'Unknown' -> Why

In [12]:
customers['city'] = customers['city'].fillna('Unknown')

**Issue 2** :     

Age has 999 Outlier -> Fill With NAN -> No such realist value

In [13]:
customers['age'] = customers['age'].replace(999, np.nan)

In [14]:
# customers.loc[customers["age"] == 999, "age"] = np.nan

**Issue 3** :

70 Null Age -> Fill with Median -> Why

In [15]:
median_age = customers['age'].median()
median_age

32.0

In [16]:
customers['age'].fillna(median_age, inplace=True)

/tmp/ipykernel_1333/651933909.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  customers['age'].fillna(median_age, inplace=True)


In [17]:
customers['age'].isnull().sum()

np.int64(0)

**Issue 3.1** :

Convert Age float to int

In [18]:
customers['age'] = customers['age'].astype('int')

In [19]:
customers['age'].dtype

dtype('int64')

**Issue 4** :

Gender has 11 unique Value -> Convert standared "Male", "Female", "Others" -> To get the clear image of genders

In [20]:
customers['gender'].unique()

array(['male', 'Female', 'Male', 'Other', 'F', 'M', 'female', 'other',
       'f', 'O', 'm'], dtype=object)

In [21]:
gender_mapping = {
    'male' : 'Male',
    'M' : 'Male',
    'm' : 'Male',
    'female' : 'Female',
    'f' : 'Female',
    'F' : 'Female',
    'other' : 'Other',
    'o' : 'Other',
    'O' : 'Other'
}

customers['gender'] = customers['gender'].replace(gender_mapping)

In [22]:
customers['gender'].unique()

array(['Male', 'Female', 'Other'], dtype=object)

**Issue 5** :

bangalore wrong spell -> Convert to Bengaluru

In [23]:
customers['city'] = customers['city'].replace('bangalore', 'Bengaluru')

## 2.4 Validate - Customers
Re-run the audit and confirm every issue in the register is resolved.

#####Issue 1 Resolve

In [24]:
customers['city'].unique()

array(['Delhi', 'Indore', 'Pune', 'Bengaluru', 'Jaipur', 'Mumbai',
       'Unknown', 'Nagpur', 'Kolkata', 'Lucknow', 'Ahmedabad',
       'Hyderabad', 'Ujjain', 'Chennai', 'Bangalore', 'Bhopal'],
      dtype=object)

#####Issue 2 Resolve

In [25]:
print(customers['age'].min())
print(customers['age'].max())

18
63


#####Issue 3 Resolve

In [26]:
customers['age'].isnull().sum()

np.int64(0)

#####Issue 3.1 Resolve

In [27]:
customers['age'].dtype

dtype('int64')

#####Issue 4 Resolve

In [28]:
customers['gender'].unique()

array(['Male', 'Female', 'Other'], dtype=object)

#####Issue 5 Resolve

In [29]:
customers['city'].value_counts(dropna=False)

,count
city,
Mumbai,153
Bengaluru,149
Delhi,145
Chennai,99
Hyderabad,95
Pune,82
Kolkata,78
Jaipur,58
Unknown,57


# 3. Cleaning - Orders table

## 3.1 Audit (find ALL problems first, fix nothing yet)

In [30]:
orders.head()

,order_id,customer_id,order_date,product_category,unit_price,quantity,discount_pct,shipping_fee,order_total,payment_method,order_status,customer_rating,delivery_days
0,O54912,C1092,2025-05-17,Fashion,555.30,1.0,30.0,0,388.71,Credit Card,Delivered,4.0,4.0
1,O52129,C1476,2025-11-09,Fashion,1966.79,1.0,10.0,0,1770.11,UPI,Delivered,4.0,3.0
2,O53815,C1779,2025-12-27,Beauty,630.90,1.0,0.0,0,630.90,Wallet,Delivered,4.0,6.0
3,O52893,C1270,2025-12-12,Beauty,658.32,1.0,10.0,0,592.49,Cash on Delivery,Returned,NaN,5.0
4,O50445,C1220,2025-02-24,Beauty,236.57,1.0,5.0,0,224.74,Credit Card,Delivered,5.0,8.0


In [31]:
orders.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6080 entries, 0 to 6079
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   order_id          6080 non-null   object 
 1   customer_id       6080 non-null   object 
 2   order_date        6080 non-null   object 
 3   product_category  6080 non-null   object 
 4   unit_price        5958 non-null   float64
 5   quantity          5896 non-null   float64
 6   discount_pct      6017 non-null   float64
 7   shipping_fee      6080 non-null   int64  
 8   order_total       6080 non-null   float64
 9   payment_method    6080 non-null   object 
 10  order_status      6080 non-null   object 
 11  customer_rating   5207 non-null   float64
 12  delivery_days     6040 non-null   float64
dtypes: float64(6), int64(1), object(6)
memory usage: 617.6+ KB


In [32]:
orders['order_date'].value_counts()

,count
order_date,
2025-10-26,32
2025-12-13,27
2025-11-06,24
2025-11-08,24
2025-10-05,23
...,...
"Oct 31, 2025",1
2023-10-26,1
2024-02-02,1


In [33]:
orders.describe()

,unit_price,quantity,discount_pct,shipping_fee,order_total,customer_rating,delivery_days
count,5958.000000,5896.000000,6017.000000,6080.000000,6080.000000,5207.000000,6040.000000
mean,2511.094483,1.639925,8.893136,5.359375,3482.780852,3.963511,4.027815
std,6587.698514,1.063430,9.084399,15.294605,6401.505233,1.137077,1.903941
min,80.000000,-1.000000,0.000000,0.000000,80.000000,1.000000,1.000000
25%,548.225000,1.000000,0.000000,0.000000,653.757500,3.000000,3.000000
50%,1118.720000,1.000000,5.000000,0.000000,1400.980000,4.000000,4.000000
75%,2397.202500,2.000000,15.000000,0.000000,3481.130000,5.000000,5.000000
max,287460.000000,5.000000,30.000000,49.000000,95010.150000,5.000000,11.000000


In [34]:
customers.shape

(1200, 8)

In [35]:
orders.describe(include='all')

,order_id,customer_id,order_date,product_category,unit_price,quantity,discount_pct,shipping_fee,order_total,payment_method,order_status,customer_rating,delivery_days
count,6080,6080,6080,6080,5958.000000,5896.000000,6017.000000,6080.000000,6080.000000,6080,6080,5207.000000,6040.000000
unique,6000,1167,1267,28,NaN,NaN,NaN,NaN,NaN,8,3,NaN,NaN
top,O50931,C1880,2025-10-26,Fashion,NaN,NaN,NaN,NaN,NaN,UPI,Delivered,NaN,NaN
freq,2,78,32,1485,NaN,NaN,NaN,NaN,NaN,2195,5207,NaN,NaN
mean,NaN,NaN,NaN,NaN,2511.094483,1.639925,8.893136,5.359375,3482.780852,NaN,NaN,3.963511,4.027815
std,NaN,NaN,NaN,NaN,6587.698514,1.063430,9.084399,15.294605,6401.505233,NaN,NaN,1.137077,1.903941
min,NaN,NaN,NaN,NaN,80.000000,-1.000000,0.000000,0.000000,80.000000,NaN,NaN,1.000000,1.000000
25%,NaN,NaN,NaN,NaN,548.225000,1.000000,0.000000,0.000000,653.757500,NaN,NaN,3.000000,3.000000
50%,NaN,NaN,NaN,NaN,1118.720000,1.000000,5.000000,0.000000,1400.980000,NaN,NaN,4.000000,4.000000
75%,NaN,NaN,NaN,NaN,2397.202500,2.000000,15.000000,0.000000,3481.130000,NaN,NaN,5.000000,5.000000


In [36]:
orders.duplicated().sum()

np.int64(79)

In [37]:
orders.isnull().sum()

,0
order_id,0
customer_id,0
order_date,0
product_category,0
unit_price,122
quantity,184
discount_pct,63
shipping_fee,0
order_total,0
payment_method,0


In [38]:
# orders.loc[
#     orders['unit_price'].isna(),
#     ['product_category', 'quantity', 'discount_pct', 'order_total', 'unit_price']
# ].head(20)

In [39]:
orders['product_category'].value_counts(dropna=False)

,count
product_category,
Fashion,1485
Electronics,969
Home & Kitchen,844
Grocery,841
Beauty,731
Sports,448
Books,404
FASHION,34
home & kitchen,29


In [40]:
orders.columns.to_list()

['order_id',
 'customer_id',
 'order_date',
 'product_category',
 'unit_price',
 'quantity',
 'discount_pct',
 'shipping_fee',
 'order_total',
 'payment_method',
 'order_status',
 'customer_rating',
 'delivery_days']

## 3.2 Issue register - Orders


| # | Column | Problem found |
|---|---|---|
| 1 | orders | order_date (mixed order) |
| 2 | orders | quntity (min : -1) |
| 3 | orders | there are outlier in unit_price |
| 4 | orders | check the total_price (error) |
| 5 | orders | product_category (repeted values) |
| 6 | orders | unit_price, dis_pct, quantity, rating, delivary_days there are missing values |
| 7 | orders | There are 79 duplicate rows |
| 8 | orders | order_date -> datetime, quantity -> int |
| 9 | orders | discount_pct has some NaN values |
| 10 | orders | there are same but different payment method |
| 11 | orders | there are nan value in discount_pct but discound is applied |


## 3.3 Fixes - Orders table

**Issue1** :

order_date (mixed order)


In [41]:
orders['order_date'] = pd.to_datetime(
    orders['order_date'],
    format = 'mixed',
    errors='coerce'
)

In [42]:
print(orders['order_date'].dtype)

datetime64[ns]


In [43]:
orders['order_date'].sample(5)

,order_date
4589,2025-09-12
2430,2025-06-28
4284,2025-06-11
5499,2024-09-22
1015,2025-11-14


**Issue2**:

Inconsistent Payment Method Values

In [44]:
orders['payment_method'].value_counts()

,count
payment_method,
UPI,2195
Credit Card,1151
Cash on Delivery,984
Debit Card,856
Net Banking,352
Wallet,289
upi,151
COD,102


In [45]:
payment_mapping = {
    'COD' : 'Cash on Delivery',
    'upi' : 'UPI'
}

orders['payment_method'] = orders['payment_method'].replace(payment_mapping)

In [46]:
orders['payment_method'].unique()

array(['Credit Card', 'UPI', 'Wallet', 'Cash on Delivery', 'Debit Card',
       'Net Banking'], dtype=object)

**Issue3**:

Inconsistent Category Values

In [47]:
orders['product_category'].nunique()

28

In [48]:
product_mapping = {
    'Fashion': 'Fashion',
    'Fashion ': 'Fashion',
    'fashion': 'Fashion',
    'FASHION': 'Fashion',

    'Beauty': 'Beauty',
    'Beauty ': 'Beauty',
    'beauty': 'Beauty',
    'BEAUTY': 'Beauty',

    'Grocery': 'Grocery',
    'Grocery ': 'Grocery',
    'grocery': 'Grocery',
    'GROCERY': 'Grocery',

    'Home & Kitchen': 'Home & Kitchen',
    'Home & Kitchen ': 'Home & Kitchen',
    'HOME & KITCHEN': 'Home & Kitchen',
    'home & kitchen': 'Home & Kitchen',

    'Sports': 'Sports',
    'Sports ': 'Sports',
    'sports': 'Sports',
    'SPORTS': 'Sports',

    'Books': 'Books',
    'Books ': 'Books',
    'books': 'Books',
    'BOOKS': 'Books',

    'Electronics': 'Electronics',
    'Electronics ': 'Electronics',
    'electronics': 'Electronics',
    'ELECTRONICS': 'Electronics'
}

orders['product_category'] = orders['product_category'].replace(product_mapping)

In [49]:
orders['product_category'].nunique()

7

**Issue4**:

79 Duplicate Rows

In [50]:
duplicates = orders[orders.duplicated(subset=['order_id'], keep=False)]

duplicates.sort_values('order_id')

,order_id,customer_id,order_date,product_category,unit_price,quantity,discount_pct,shipping_fee,order_total,payment_method,order_status,customer_rating,delivery_days
1056,O50079,C1566,2025-07-04,Electronics,7314.89,1.0,5.0,0,6949.15,Debit Card,Delivered,5.0,1.0
1002,O50079,C1566,2025-07-04,Electronics,7314.89,1.0,5.0,0,6949.15,Debit Card,Delivered,5.0,1.0
2759,O50125,C1300,2025-06-28,Home & Kitchen,859.18,4.0,5.0,0,3264.88,UPI,Delivered,5.0,4.0
599,O50125,C1300,2025-06-28,Home & Kitchen,859.18,4.0,5.0,0,3264.88,UPI,Delivered,5.0,4.0
2163,O50179,C1306,2025-06-16,Electronics,11938.56,2.0,15.0,0,20295.55,Credit Card,Delivered,2.0,9.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
992,O55908,C1703,2025-12-03,Electronics,4501.21,3.0,0.0,0,13503.63,Cash on Delivery,Cancelled,NaN,5.0
1571,O55962,C1196,2025-06-17,Fashion,421.11,2.0,0.0,0,842.22,Credit Card,Delivered,4.0,4.0
3167,O55962,C1196,2025-06-17,Fashion,421.11,2.0,0.0,0,842.22,Credit Card,Delivered,4.0,4.0
4611,O55988,C1516,2025-11-13,Fashion,2163.69,1.0,20.0,0,1730.95,Wallet,Delivered,5.0,3.0


In [51]:
orders = orders.drop_duplicates()

In [52]:
orders.duplicated().sum()

np.int64(0)

**Issue5**:

Fill NaN to median day iin Delivered and Returned order status

In [53]:
# orders.loc[orders['order_status'] == 'Returned', ['order_status', 'delivery_days']]

orders.loc[orders['delivery_days'].isna(), ['order_status', 'delivery_days']].head()

,order_status,delivery_days
218,Delivered,NaN
395,Returned,NaN
442,Delivered,NaN
670,Delivered,NaN
869,Delivered,NaN


In [54]:
median_days = orders['delivery_days'].median()
mask = orders['delivery_days'].isna() & (orders['order_status'] != 'Cancelled')
orders.loc[mask, 'delivery_days'] = median_days

In [55]:
orders.loc[orders['delivery_days'].isna(), ['order_status', 'delivery_days']]

,order_status,delivery_days
2902,Cancelled,NaN


**Issute6**:

Fix the quantity values

In [56]:
orders.loc[orders['quantity'] < 0, 'quantity'] = np.nan

In [57]:
orders['quantity'].isnull().sum()

np.int64(205)

In [58]:
# order_total = unit_price × quantity × (1 − discount_pct/100) + shipping_fee

In [59]:
calculated_quantity = (
    (orders['order_total'] - orders['shipping_fee'])
    / (
        orders['unit_price']
        * (1 - orders['discount_pct'] / 100)
    )
)

In [60]:
orders.loc[orders['quantity'].isnull(), 'quantity'] = calculated_quantity

In [61]:
orders['quantity'].isnull().sum()

np.int64(9)

In [62]:
orders[orders['quantity'].isnull()]

,order_id,customer_id,order_date,product_category,unit_price,quantity,discount_pct,shipping_fee,order_total,payment_method,order_status,customer_rating,delivery_days
213,O51320,C1775,2025-10-16,Sports,3459.00,NaN,NaN,0,6918.00,Wallet,Delivered,4.0,5.0
683,O52092,C1623,2024-11-25,Beauty,NaN,NaN,0.0,0,1690.11,UPI,Delivered,2.0,4.0
1819,O55126,C1540,2025-08-20,Home & Kitchen,NaN,NaN,5.0,0,2677.38,Net Banking,Delivered,4.0,5.0
2413,O54546,C1787,2025-09-22,Electronics,9624.99,NaN,NaN,0,43312.46,Credit Card,Delivered,5.0,1.0
2899,O50268,C1223,2023-07-31,Beauty,488.89,NaN,NaN,0,1393.34,Credit Card,Delivered,5.0,5.0
2928,O55526,C2171,2025-12-03,Beauty,NaN,NaN,0.0,0,351.32,Credit Card,Delivered,1.0,4.0
3506,O52098,C1785,2025-07-12,Books,NaN,NaN,15.0,0,719.95,Debit Card,Delivered,5.0,3.0
4736,O53160,C1869,2024-11-14,Beauty,764.95,NaN,NaN,0,688.46,Debit Card,Delivered,5.0,3.0
5933,O53332,C1085,2025-08-24,Books,100.00,NaN,NaN,0,240.00,Credit Card,Delivered,2.0,4.0


**Issue7**:

Fix the unit_price values

In [63]:
orders['unit_price'].isnull().sum()

np.int64(120)

In [64]:
calculated_unit_price = (
    (orders['order_total'] - orders['shipping_fee'])
    /
    (
        orders['quantity']
        * (1 - orders['discount_pct'] / 100)
    )
)

In [65]:
orders.loc[orders['unit_price'].isnull(), 'unit_price'] = calculated_unit_price

In [66]:
orders['unit_price'].isnull().sum()

np.int64(4)

In [67]:
orders[orders['unit_price'].isnull()]

,order_id,customer_id,order_date,product_category,unit_price,quantity,discount_pct,shipping_fee,order_total,payment_method,order_status,customer_rating,delivery_days
683,O52092,C1623,2024-11-25,Beauty,NaN,NaN,0.0,0,1690.11,UPI,Delivered,2.0,4.0
1819,O55126,C1540,2025-08-20,Home & Kitchen,NaN,NaN,5.0,0,2677.38,Net Banking,Delivered,4.0,5.0
2928,O55526,C2171,2025-12-03,Beauty,NaN,NaN,0.0,0,351.32,Credit Card,Delivered,1.0,4.0
3506,O52098,C1785,2025-07-12,Books,NaN,NaN,15.0,0,719.95,Debit Card,Delivered,5.0,3.0


**Issue8**:

Fix the discount_pct values

In [68]:
orders['discount_pct'].isnull().sum()

np.int64(60)

In [69]:
calculated_discount_pct = (
    1 - (
        (orders['order_total'] - orders['shipping_fee'])
        / (orders['unit_price'] * orders['quantity'])
    )
) * 100

In [70]:
orders.loc[orders['discount_pct'].isnull(), 'discount_pct'] = calculated_discount_pct

In [71]:
orders['discount_pct'].isnull().sum()

np.int64(5)

In [72]:
orders[orders['discount_pct'].isnull()]

,order_id,customer_id,order_date,product_category,unit_price,quantity,discount_pct,shipping_fee,order_total,payment_method,order_status,customer_rating,delivery_days
213,O51320,C1775,2025-10-16,Sports,3459.00,NaN,NaN,0,6918.00,Wallet,Delivered,4.0,5.0
2413,O54546,C1787,2025-09-22,Electronics,9624.99,NaN,NaN,0,43312.46,Credit Card,Delivered,5.0,1.0
2899,O50268,C1223,2023-07-31,Beauty,488.89,NaN,NaN,0,1393.34,Credit Card,Delivered,5.0,5.0
4736,O53160,C1869,2024-11-14,Beauty,764.95,NaN,NaN,0,688.46,Debit Card,Delivered,5.0,3.0
5933,O53332,C1085,2025-08-24,Books,100.00,NaN,NaN,0,240.00,Credit Card,Delivered,2.0,4.0


**Issue9**:

There are some null values

In [73]:
orders.loc[
    orders[['unit_price', 'quantity', 'discount_pct', 'shipping_fee', 'order_total']].isnull().any(axis=1),
           ['unit_price', 'quantity', 'discount_pct', 'shipping_fee', 'order_total']
]

,unit_price,quantity,discount_pct,shipping_fee,order_total
213,3459.00,NaN,NaN,0,6918.00
683,NaN,NaN,0.0,0,1690.11
1819,NaN,NaN,5.0,0,2677.38
2413,9624.99,NaN,NaN,0,43312.46
2899,488.89,NaN,NaN,0,1393.34
2928,NaN,NaN,0.0,0,351.32
3506,NaN,NaN,15.0,0,719.95
4736,764.95,NaN,NaN,0,688.46
5933,100.00,NaN,NaN,0,240.00


**Issue10**:

There are outliers in unit_price

In [74]:
unit_price_outliers = orders['unit_price'].quantile(0.995)
orders[orders['unit_price'] > unit_price_outliers]

,order_id,customer_id,order_date,product_category,unit_price,quantity,discount_pct,shipping_fee,order_total,payment_method,order_status,customer_rating,delivery_days
252,O51464,C1889,2024-06-09,Sports,75055.6,4.000,0.0,0,7505.56,UPI,Delivered,3.0,6.0
380,O50615,C1340,2025-09-19,Home & Kitchen,33506.0,1.000,15.0,0,712.00,UPI,Delivered,5.0,3.0
951,O55534,C1232,2023-10-20,Fashion,118856.0,2.000,0.0,0,5942.80,Debit Card,Delivered,5.0,5.0
1331,O52635,C2029,2025-10-25,Electronics,287460.0,0.025,0.0,0,7186.50,UPI,Returned,NaN,3.0
2232,O50821,C1196,2024-05-16,Sports,124622.0,1.000,15.0,0,2648.22,Debit Card,Delivered,5.0,6.0
2553,O53182,C2127,2025-06-26,Beauty,32550.0,1.000,0.0,0,813.75,Debit Card,Delivered,5.0,7.0
3180,O55730,C1317,2025-03-19,Fashion,126846.8,3.000,0.0,0,9513.51,Debit Card,Delivered,2.0,4.0
3529,O50270,C1734,2024-05-12,Electronics,144292.0,5.000,0.0,0,18036.50,Credit Card,Delivered,4.0,5.0
4283,O55777,C1644,2025-07-05,Beauty,36578.0,1.000,15.0,0,777.28,Credit Card,Returned,NaN,5.0
4487,O53630,C1480,2024-06-30,Sports,172230.4,2.000,20.0,0,6889.22,Cash on Delivery,Cancelled,NaN,5.0


In [75]:
original_unit_price = (
    (orders['order_total'] - orders['shipping_fee']) /
    (orders['quantity'] * (1 - orders['discount_pct'] / 100))
)

In [76]:
orders.loc[orders['unit_price'] > orders['unit_price'].quantile(0.995), 'unit_price'] = original_unit_price

In [77]:
unit_price_outliers = orders['unit_price'].quantile(0.995)
orders[orders['unit_price'] > unit_price_outliers]

,order_id,customer_id,order_date,product_category,unit_price,quantity,discount_pct,shipping_fee,order_total,payment_method,order_status,customer_rating,delivery_days
18,O51355,C1960,2024-03-20,Electronics,25000.00,4.000,10.0,0,90000.00,Cash on Delivery,Delivered,5.0,7.0
234,O55336,C2039,2025-10-18,Electronics,25000.00,2.000,0.0,0,50000.00,Credit Card,Delivered,5.0,6.0
861,O54133,C1204,2025-10-22,Electronics,21817.72,1.000,0.0,0,21817.72,UPI,Delivered,5.0,7.0
888,O50078,C1044,2023-06-25,Electronics,22119.96,1.000,0.0,0,22119.96,UPI,Delivered,3.0,8.0
1035,O53015,C1473,2025-12-26,Electronics,23625.21,3.000,30.0,0,49612.94,Wallet,Cancelled,NaN,1.0
1331,O52635,C2029,2025-10-25,Electronics,287460.00,0.025,0.0,0,7186.50,UPI,Returned,NaN,3.0
1447,O54673,C1604,2025-08-20,Electronics,25000.00,3.000,0.0,0,75000.00,UPI,Delivered,4.0,2.0
1784,O54044,C1204,2025-10-21,Electronics,25000.00,1.000,0.0,0,25000.00,UPI,Delivered,5.0,7.0
1787,O51647,C1935,2025-12-22,Electronics,25000.00,2.000,20.0,0,40000.00,Net Banking,Delivered,4.0,5.0
1859,O50492,C1754,2024-08-11,Electronics,25000.00,2.000,5.0,0,47500.00,Debit Card,Delivered,5.0,4.0


**Issue11**:

Fix all float quantity to appropriate value

In [78]:
# Get quantities that are not whole numbers
decimal_quantities = orders[orders['quantity'] % 1 != 0]

print(decimal_quantities['quantity'].unique())

[0.99999967 1.00000095 1.00000675 1.00001409 1.99999873        nan
 1.00001531 1.00001672 0.9999996  1.0000015  2.9999977  0.99999669
 1.9999971  1.9999929  1.00000071 0.99999955 1.00002815 1.0000026
 1.00000127 0.99999085 3.00000581 4.00000043 2.00000079 1.00000241
 0.9999997  0.025      1.99999791 0.99999402 2.00000154 1.00000895
 1.00002276 1.00000387 1.00000785 0.99999974 2.00000363 1.99999908
 1.99999532 1.00000376 1.00000185 0.99999872 1.00000162 2.99999164
 0.99999725 1.99999577 1.00000355 1.0000007  1.         1.99999761
 1.00000136 0.9999767  0.99998789 0.99999564 1.000007   1.00000282
 2.00000577 1.00000084 0.99999449 1.00000115 1.00000292 1.00003492
 1.00000053 1.99999963 3.00000083 2.99999879 0.99999789 1.99999438
 2.00001408 1.00001278 0.99998485 2.00000025 1.         0.99999927
 1.000005   3.00000725 1.00000214 1.00001668 0.99999324 2.99999729
 1.00000082 1.99999639 1.99997398 0.99999448 0.99999497 0.999996
 0.9999993  0.99999063 0.99999128 0.99999937 1.00000091 0.9999925

In [79]:
orders.loc[orders['quantity'] % 1 != 0]

,order_id,customer_id,order_date,product_category,unit_price,quantity,discount_pct,shipping_fee,order_total,payment_method,order_status,customer_rating,delivery_days
7,O51677,C1187,2025-11-22,Sports,3230.58,1.000000,5.0,0,3069.05,UPI,Delivered,5.0,4.0
77,O54643,C1837,2025-03-17,Fashion,1167.81,1.000001,10.0,0,1051.03,Cash on Delivery,Delivered,1.0,5.0
83,O51715,C1409,2025-11-24,Grocery,370.46,1.000007,20.0,49,345.37,Credit Card,Delivered,5.0,6.0
124,O51580,C1713,2025-09-29,Grocery,405.68,1.000014,30.0,0,283.98,Credit Card,Delivered,2.0,7.0
139,O50149,C2134,2024-11-29,Fashion,1753.69,1.999999,10.0,0,3156.64,UPI,Delivered,3.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
5933,O53332,C1085,2025-08-24,Books,100.00,NaN,NaN,0,240.00,Credit Card,Delivered,2.0,4.0
5954,O50726,C1980,2025-04-30,Fashion,1049.11,1.000004,30.0,0,734.38,UPI,Delivered,4.0,5.0
6022,O51685,C2161,2025-09-29,Home & Kitchen,1279.99,1.999998,10.0,0,2303.98,UPI,Delivered,1.0,1.0
6041,O51161,C1085,2025-09-23,Home & Kitchen,1191.32,4.000004,5.0,0,4527.02,Credit Card,Returned,NaN,8.0


In [80]:
orders.loc[orders['quantity'] % 1 != 0, 'quantity'] = decimal_quantities['quantity'].round()

In [81]:
orders[orders['quantity'] % 1 != 0]

,order_id,customer_id,order_date,product_category,unit_price,quantity,discount_pct,shipping_fee,order_total,payment_method,order_status,customer_rating,delivery_days
213,O51320,C1775,2025-10-16,Sports,3459.00,NaN,NaN,0,6918.00,Wallet,Delivered,4.0,5.0
683,O52092,C1623,2024-11-25,Beauty,NaN,NaN,0.0,0,1690.11,UPI,Delivered,2.0,4.0
1819,O55126,C1540,2025-08-20,Home & Kitchen,NaN,NaN,5.0,0,2677.38,Net Banking,Delivered,4.0,5.0
2413,O54546,C1787,2025-09-22,Electronics,9624.99,NaN,NaN,0,43312.46,Credit Card,Delivered,5.0,1.0
2899,O50268,C1223,2023-07-31,Beauty,488.89,NaN,NaN,0,1393.34,Credit Card,Delivered,5.0,5.0
2928,O55526,C2171,2025-12-03,Beauty,NaN,NaN,0.0,0,351.32,Credit Card,Delivered,1.0,4.0
3506,O52098,C1785,2025-07-12,Books,NaN,NaN,15.0,0,719.95,Debit Card,Delivered,5.0,3.0
4736,O53160,C1869,2024-11-14,Beauty,764.95,NaN,NaN,0,688.46,Debit Card,Delivered,5.0,3.0
5933,O53332,C1085,2025-08-24,Books,100.00,NaN,NaN,0,240.00,Credit Card,Delivered,2.0,4.0


We almost fix all the issues from the orders table there would be the missing values so our formula not able to predict the value of Null values so we leave them NaN

# 4 Save Clean Data

In [82]:
customers.to_csv('customers_clean.csv', index=False)
orders.to_csv('orders_clean.csv', index=False)